# 21 · Final tables and figures from saved results (~7 min, no mining)

Regenerates every table, figure and quoted number of the study from the saved campaigns: the May 2026 grid, `vm_20261029` and `vm_20261031`. It needs no data files and no mining. Outputs go to `results/derived/final/`: LaTeX table rows and CSVs in `tables/`, every quoted number in `tables/numbers.json`, and figures in `figures/`.

Part 1 is the regime diagnostic (threshold sweep, importance, leave-one-dataset-out), with the 32 failure cells in `tables/ml_imputed_cells.csv`. Part 2 builds the rest and asserts the checks stated in the study, such as count agreement and cell totals.

Names: `P1` in the saved files is the indicator written rho-intersection, and `rho` is rho-delta. `fastapriori_fast` and `hyb` are the anchor-and-extend path, `hyb_seeded` is scatter-seeded Eclat.

On many-core Linux set `OMP_THREAD_LIMIT=4 OMP_WAIT_POLICY=PASSIVE` for the diagnostic.


In [1]:
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
OUT = BUNDLE / "results" / "derived" / "final"
GEN, FIG = OUT / "tables", OUT / "figures"
GEN.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)


In [2]:
from math import factorial
import json
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold

warnings.filterwarnings('ignore')
ART = BUNDLE / 'results' / 'campaign_2026-05'
LIMIT_S, PLACEHOLDER_S = 1500.0, 500.0
LOSS_ON_OOM = {'Online Retail', 'Kosarak', 'BMS-WebView-1'}
SEEDS = list(range(42, 52))
THRESHOLDS = range(1, 11)
PYFIM = ['pyfim_apriori', 'pyfim_eclat', 'pyfim_fpgrowth']

agg = pd.read_csv(ART / 'results_rigorous_20260523_115442.csv', low_memory=False)
ok = agg[agg.wall_s_n_ok > 0]
assert len(ok) == 4891
times = ok.pivot_table(index=['dataset', 'k', 'min_support'], columns='method', values='wall_s_median').reset_index()
times['pyfim_time'] = times[PYFIM].min(axis=1)
times = times.rename(columns={'fastapriori_fast': 'fast_time', 'fastapriori_classic': 'classic_time'})

ds = agg.groupby('dataset').agg(n_transactions=('n_transactions', 'first'), n_items=('n_items', 'first'),
                                n_rows=('n_rows', 'first'), d_avg=('avg_items_per_txn', 'first'),
                                d_max=('max_items_per_txn', 'first'), d_std=('std_items_per_txn', 'first'),
                                density=('density', 'first')).reset_index()
ds['d_CV'] = ds.d_std / ds.d_avg
ds['d_max_by_d_avg'] = ds.d_max / ds.d_avg
ds['I_est'] = ds.d_avg.apply(lambda x: max(2, int(round(x / 2.5))))
ds['L_est'] = ds.n_items.apply(lambda x: max(10, int(round(2 * x))))


In [3]:
def impute_classic(times):
    """Failed cells of the anchor/classic comparison.

    A failed side takes the 1,500 s limit. On the three LOSS_ON_OOM datasets an anchor OOM counts as a loss to
    classic: anchor takes the limit and classic its measured time, or a 500 s placeholder when it has none.
    """
    cov = pd.read_csv(ART / 'coverage_campaign_2026-05.csv')
    s = cov[cov.method.isin(['fastapriori_fast', 'fastapriori_classic'])].pivot_table(
        index=['dataset', 'k', 'min_support'], columns='method', values='probe_status', aggfunc='first')
    measured = times.set_index(['dataset', 'k', 'min_support'])
    out = []
    for (d, k, ms), r in s.iterrows():
        f, c = r.fastapriori_fast, r.fastapriori_classic
        f_bad, c_bad = f in ('oom', 'timeout'), c in ('oom', 'timeout')
        if not (f_bad or c_bad):
            continue
        real = measured.classic_time.get((d, k, ms), np.nan)
        if d in LOSS_ON_OOM:
            fast, classic = LIMIT_S, real if pd.notna(real) else PLACEHOLDER_S
            kind = 'measured' if pd.notna(real) else 'placeholder'
        else:
            fast = LIMIT_S if f_bad else measured.fast_time[(d, k, ms)]
            classic, kind = LIMIT_S if c_bad else real, 'limit'
        out.append(dict(dataset=d, k=k, min_support=ms, fast_time=fast, classic_time=classic, kind=kind, fast_status=f, classic_status=c))
    return pd.DataFrame(out)


imputed = impute_classic(times)
assert len(imputed) == 32
imputed.to_csv(GEN / 'ml_imputed_cells.csv', index=False)
print(imputed.groupby(['dataset', 'kind']).size())
keep = ['dataset', 'k', 'min_support', 'fast_time', 'classic_time']
times_imp = pd.concat([times[keep].dropna(subset=['fast_time', 'classic_time']), imputed[keep]], ignore_index=True)
assert len(times_imp) == 574
times_imp = times_imp.rename(columns={'classic_time': 'classic_imp_time', 'fast_time': 'fast_imp_time'})
times = times.merge(times_imp, on=['dataset', 'k', 'min_support'], how='outer')
times = times.merge(ds, on='dataset', how='left')
times['log_D'] = np.log1p(times.n_transactions)
times['log_N'] = np.log1p(times.n_items)
times['log_rows'] = np.log1p(times.n_rows)
times['log_d_avg'] = np.log1p(times.d_avg)
times['log_support'] = np.log1p(times.min_support)
times['log_I'] = np.log1p(times.I_est)
times['log_L'] = np.log1p(times.L_est)
times['k_x_d_avg'] = times.k * times.d_avg
times['k_x_log_D'] = times.k * times.log_D
times['d_avg_x_log_D'] = times.d_avg * times.log_D
times['log_comb_cost'] = np.log1p(times.d_avg ** times.k / times.k.apply(factorial))
times['txn_per_item'] = times.n_transactions / (times.n_items + 1)
FULL = ['log_D', 'log_N', 'log_rows', 'log_d_avg', 'log_support', 'k', 'd_avg', 'd_max', 'd_std', 'log_I', 'log_L',
        'k_x_d_avg', 'k_x_log_D', 'd_avg_x_log_D', 'density', 'log_comb_cost', 'txn_per_item', 'd_max_by_d_avg', 'd_CV']
assert len(FULL) == 19


def model(seed):
    return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                             random_state=seed, n_jobs=4, eval_metric='logloss')


def random_folds(X, y, seed):
    prob = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        prob[te] = model(seed).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return prob


def logo(X, y, groups, seed):
    prob = np.zeros(len(y))
    for tr, te in LeaveOneGroupOut().split(X, y, groups):
        if len(set(y[tr])) < 2:
            prob[te] = y[tr].mean()
            continue
        prob[te] = model(seed).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return prob


dataset                  kind       
BMS-WebView-1            placeholder     3
Groceries                limit           3
Kosarak                  measured        1
                         placeholder     4
Online Retail            placeholder     9
T25.I8.D5000k.N40k.L40k  limit          12
dtype: int64


In [4]:
sweeps, summary, importance = [], [], {}
for task, fc, col in [('pyfim', 'fast_time', 'pyfim_time'), ('classic', 'fast_imp_time', 'classic_imp_time'), ('classic_measured', 'fast_time', 'classic_time')]:
    t = times.dropna(subset=[fc, col] + FULL).reset_index(drop=True)
    X, g = t[FULL].values, t.dataset.values
    gap = (t[col] - t[fc]).values
    for T in THRESHOLDS:
        y = (gap > -T).astype(int)
        if len(set(y)) < 2 or min(np.bincount(y)) < 5:
            continue
        p = [random_folds(X, y, s) for s in SEEDS]
        sweeps.append(dict(task=task, T=T, n=len(y), n_pos=int(y.sum()), n_neg=int(len(y) - y.sum()),
                           mcc=np.mean([matthews_corrcoef(y, q > .5) for q in p]),
                           auroc=np.mean([roc_auc_score(y, q) for q in p]),
                           acc=np.mean([accuracy_score(y, q > .5) for q in p])))
        print(json.dumps(sweeps[-1], default=float), flush=True)
    sw = pd.DataFrame([r for r in sweeps if r['task'] == task])
    star = int(sw.loc[sw.mcc.idxmax(), 'T'])
    y = (gap > -star).astype(int)
    ps = np.array([logo(X, y, g, s) for s in SEEDS])
    m = ps.mean(axis=0)
    folds = [roc_auc_score(y[g == d], m[g == d]) for d in np.unique(g) if len(set(y[g == d])) == 2]
    summary.append(dict(task=task, T_star=star, n=len(y), n_datasets=len(set(g)), positive=float(y.mean()),
                        cv_mcc=float(sw.mcc.max()), cv_auroc=float(sw.loc[sw.mcc.idxmax(), 'auroc']),
                        logo_mcc=np.mean([matthews_corrcoef(y, q > .5) for q in ps]),
                        logo_auroc=np.mean([roc_auc_score(y, q) for q in ps]),
                        logo_acc=np.mean([accuracy_score(y, q > .5) for q in ps]),
                        majority_acc=float(max(y.mean(), 1 - y.mean())),
                        macro_auroc=float(np.mean(folds)), folds_two_classes=len(folds)))
    print(json.dumps(summary[-1], default=float), flush=True)
    importance[task] = pd.Series(np.mean([model(s).fit(X, y).feature_importances_ for s in SEEDS], axis=0), index=FULL)

pd.DataFrame(sweeps).to_csv(GEN / 'ml_threshold_sweep.csv', index=False)
pd.DataFrame(summary).to_csv(GEN / 'ml_summary.csv', index=False)
pd.DataFrame(importance).rename_axis('feature').to_csv(GEN / 'ml_importance.csv')

# Why density and d_max: Pearson correlations of the top features across the 18 datasets.
f = ds.set_index('dataset')
c = pd.DataFrame({'log_dmax': np.log(f.d_max), 'log_density': np.log(f.density), 'log_davg': np.log(f.d_avg),
                  'log_rows': np.log(f.n_rows), 'log_N': np.log(f.n_items), 'log_D': np.log(f.n_transactions)})
c.corr('pearson').to_csv(GEN / 'ml_feature_correlation.csv')
print(c.corr('pearson').round(3))
print(pd.DataFrame(importance).round(3).sort_values('pyfim', ascending=False))


{"task": "pyfim", "T": 1, "n": 591, "n_pos": 460, "n_neg": 131, "mcc": 0.9247966672004235, "auroc": 0.996178227680053, "acc": 0.9742808798646362}


{"task": "pyfim", "T": 2, "n": 591, "n_pos": 480, "n_neg": 111, "mcc": 0.8763536433573116, "auroc": 0.9861524024024024, "acc": 0.9629441624365483}


{"task": "pyfim", "T": 3, "n": 591, "n_pos": 514, "n_neg": 77, "mcc": 0.7809613592315021, "auroc": 0.9781242104199303, "acc": 0.9531302876480542}


{"task": "pyfim", "T": 4, "n": 591, "n_pos": 529, "n_neg": 62, "mcc": 0.7901830853267098, "auroc": 0.9854472833709373, "acc": 0.962605752961083}


{"task": "pyfim", "T": 5, "n": 591, "n_pos": 543, "n_neg": 48, "mcc": 0.7528492092255187, "auroc": 0.9776051258440761, "acc": 0.9661590524534687}


{"task": "pyfim", "T": 6, "n": 591, "n_pos": 549, "n_neg": 42, "mcc": 0.7661002633935846, "auroc": 0.9574767976407321, "acc": 0.971404399323181}


... (24 outputs left out)

{"task": "classic_measured", "T": 9, "n": 542, "n_pos": 537, "n_neg": 5, "mcc": -0.0008297155366092726, "auroc": 0.8286219739292365, "acc": 0.9904059040590406}


{"task": "classic_measured", "T": 10, "n": 542, "n_pos": 537, "n_neg": 5, "mcc": -0.0008297155366092726, "auroc": 0.8286219739292365, "acc": 0.9904059040590406}


{"task": "classic_measured", "T_star": 2, "n": 542, "n_datasets": 18, "positive": 0.985239852398524, "cv_mcc": 0.31851499838907005, "cv_auroc": 0.9075257490636703, "logo_mcc": -0.005918282839289827, "logo_auroc": 0.8753511235955056, "logo_acc": 0.9828413284132843, "majority_acc": 0.985239852398524, "macro_auroc": 0.9194426283099052, "folds_two_classes": 5}


             log_dmax  log_density  log_davg  log_rows  log_N  log_D
log_dmax        1.000       -0.243    -0.010     0.264  0.234  0.314
log_density    -0.243        1.000     0.143    -0.524 -0.919 -0.669
log_davg       -0.010        0.143     1.000     0.591  0.259  0.322
log_rows        0.264       -0.524     0.591     1.000  0.747  0.954
log_N           0.234       -0.919     0.259     0.747  1.000  0.781
log_D           0.314       -0.669     0.322     0.954  0.781  1.000
                pyfim  classic  classic_measured
log_D           0.139    0.043             0.043
log_rows        0.116    0.030             0.168
d_max           0.109    0.226             0.055
density         0.082    0.030             0.067
log_support     0.067    0.110             0.077
d_std           0.058    0.042             0.026
log_N           0.052    0.022             0.073
log_L           0.047    0.009             0.101
k               0.043    0.084             0.084
d_max_by_d_avg  0.041    0.

In [5]:
import ast
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ART = BUNDLE / 'results'
MAY = ART / 'campaign_2026-05'
OCT = ART / 'runs' / 'vm_20261029'
NOV = ART / 'runs' / 'vm_20261031'
DER = ART / 'derived'
short = {'T3.I2.D100k.N50k.L50k': 'T3.D100k', 'T5.I2.D25k.N1k.L1k': 'T5.D25k', 'T10.I4.D100k.N25k.L25k': 'T10.D100k', 'T12.I4.D5k.N1k.L1k': 'T12.D5k',
         'T15.I8.D50k.N1k.L1k.high': 'T15.D50k.h', 'T15.I8.D50k.N1k.L1k.low': 'T15.D50k.l', 'T20.I4.D1000k.N50k.L50k': 'T20.D1M', 'T25.I8.D5000k.N40k.L40k': 'T25.D5M',
         'T30.I8.D2000k.N10k.L10k': 'T30.D2M', 'T30.I8.D50k.N5k.L5k': 'T30.D50k', 'Retail (Belgian)': 'Retail'}
def name(d): return short.get(d, d)
def num(v, digits=2): return '---' if pd.isna(v) else f'{v:.{digits}f}'
def integer(v): return f'{int(v):,}'.replace(',', '{,}')
def rows(file, lines): (GEN / file).write_text('\n'.join(lines) + '\n', encoding='utf-8')
def rng(s, digits=2): return f'{s.min():.{digits}f}--{s.max():.{digits}f}'
N = {}                                                     # every number quoted in the prose

EXT = ['borgelt_apriori', 'borgelt_eclat', 'borgelt_fpgrowth', 'pyfim_apriori', 'pyfim_eclat', 'pyfim_fpgrowth']
raw = pd.read_csv(MAY / 'results_rigorous_20260523_115442.csv', low_memory=False)
ok = raw[raw.wall_s_n_ok > 0].copy()
assert len(ok) == 4891 and ok.dataset.nunique() == 18 and (ok.wall_s_n_ok == 3).all()
cov = pd.read_csv(MAY / 'coverage_campaign_2026-05.csv')
stats = pd.read_csv(MAY / 'cost_stats_20261022.csv')
st = stats[np.isclose(stats.min_support, 1e-4)].set_index('dataset')
N['attempted'] = len(cov)
N['outcomes'] = cov.probe_status.value_counts().to_dict()
N['outcomes_by_method'] = cov[cov.probe_status != 'ok'].groupby(['method', 'probe_status']).size().to_dict()
N['outcomes_by_method'] = {' '.join(k): int(v) for k, v in N['outcomes_by_method'].items()}


In [6]:
# ---- count agreement, May campaign -------------------------------------------------------------
g = ok[ok.method != 'efficient_apriori'].groupby(['dataset', 'k', 'min_support'])
agree = g.n_itemsets_median.agg(['nunique', 'count'])
agree = agree[agree['count'] >= 2]
N['may_agreement_cells'], N['may_agreement_conflicts'] = len(agree), int((agree['nunique'] > 1).sum())
assert N['may_agreement_cells'] == 614 and N['may_agreement_conflicts'] == 0


In [7]:
# ---- pairs at s=1e-4 -----------------------------------------------------------------------------
lo = ok[np.isclose(ok.min_support, 1e-4)]
w = lo.pivot_table(index=['dataset', 'k'], columns='method', values='wall_s_median')
pair = w.xs(2, level='k')
best = pair[EXT].min(axis=1)
N['k2_8t_wins'] = int((best > pair.fastapriori_fast).sum())
N['k2_8t_median'] = float((best / pair.fastapriori_fast).median())
N['k2_1t_wins'] = int((best > pair.fastapriori_fast_1t).sum())
N['k2_1t_median'] = float((best / pair.fastapriori_fast_1t).median())
eclat = pair[['pyfim_eclat', 'borgelt_eclat']].min(axis=1)
N['k2_8t_vs_eclat_wins'] = int((eclat > pair.fastapriori_fast).sum())
N['k2_8t_vs_eclat_median'] = float((eclat / pair.fastapriori_fast).median())
cl = pair.dropna(subset=['fastapriori_classic'])
N['k2_8t_vs_classic_wins'], N['k2_classic_n'] = int((cl.fastapriori_classic > cl.fastapriori_fast).sum()), len(cl)
N['k2_8t_vs_classic_median'] = float((cl.fastapriori_classic / cl.fastapriori_fast).median())
N['k2_1t_vs_classic_median'] = float((cl.fastapriori_classic / cl.fastapriori_fast_1t).median())
N['k2_1t_vs_classic_range'] = [float((cl.fastapriori_classic / cl.fastapriori_fast_1t).min()), float((cl.fastapriori_classic / cl.fastapriori_fast_1t).max())]
par = pair.fastapriori_fast_1t / pair.fastapriori_fast
N['parallel_gain'] = dict(median=float(par.median()), min=float(par.min()), max=float(par.max()), below_1_3=int((par < 1.3).sum()))
slow = best >= .1
N['k2_8t_wins_resolved'] = [int((best[slow] > pair.fastapriori_fast[slow]).sum()), int(slow.sum())]


In [8]:
# ---- cost model ----------------------------------------------------------------------------------
cx = st.join(pair[['fastapriori_classic', 'fastapriori_fast_1t', 'fastapriori_fast', 'borgelt_apriori']])
cx['c_1t'] = cx.fastapriori_classic / cx.fastapriori_fast_1t
cx['c_8t'] = cx.fastapriori_classic / cx.fastapriori_fast
cx['b_1t'] = cx.borgelt_apriori / cx.fastapriori_fast_1t
def spearman(q, a, b): return q[[a, b]].dropna().corr('spearman').iloc[0, 1]
def loo(q, a, b):
    q = q[[a, b]].dropna()
    r = [spearman(q.drop(i), a, b) for i in q.index]
    return [float(min(r)), float(max(r))]
N['sp_1t'], N['sp_8t'] = float(spearman(cx, 'P1_raw', 'c_1t')), float(spearman(cx, 'P1_raw', 'c_8t'))
assert abs(N['sp_1t'] - .946078431372549) < 1e-12 and round(N['sp_8t'], 3) == .924
N['sp_1t_loo'], N['sp_8t_loo'] = loo(cx, 'P1_raw', 'c_1t'), loo(cx, 'P1_raw', 'c_8t')
N['sp_rho_borgelt'] = float(spearman(cx, 'rho_raw', 'b_1t'))
N['P1_range_sparse'] = [float(cx.P1_raw.min()), float(cx.P1_raw.max())]
N['density_max_sparse'] = float((st.d_avg / st.N).max())
N['c8t_range'] = [float(cx.c_8t.min()), float(cx.c_8t.max())]
N['rho_ge_09'] = [int(((cx.rho_raw >= .9) & (cx.b_1t > 1)).sum()), int((cx.rho_raw >= .9).sum())]
N['rho_le_03'] = [int(((cx.rho_raw <= .3) & (cx.b_1t < 1)).sum()), int((cx.rho_raw <= .3).sum())]


In [9]:
# ---- dense data ----------------------------------------------------------------------------------
fx = pd.read_csv(OCT / 'fimi_lite_crossover.csv')
fx['best_b_8t'] = fx[['borgelt_apriori', 'borgelt_eclat']].min(axis=1) / fx.hyb
fx['best_b_1t'] = fx[['borgelt_apriori', 'borgelt_eclat']].min(axis=1) / fx.hyb_1t
res = fx[fx['classic/scatter 1T resolved']]
N['dense_cells'], N['dense_resolved'] = len(fx), len(res)
N['dense_sp_1t'] = float(res[['P1', 'classic/scatter 1T']].corr('spearman').iloc[0, 1])
N['dense_sp_8t'] = float(res[['P1', 'classic/scatter 8T']].corr('spearman').iloc[0, 1])
N['dense_min_ratio_1t'] = float(fx['classic/scatter 1T'].min())
N['dense_min_ratio_8t'] = float(fx['classic/scatter 8T'].min())
N['dense_max_ratio_1t'] = float(fx['classic/scatter 1T'].max())
N['dense_max_ratio_8t'] = float(fx['classic/scatter 8T'].max())
N['dense_P1_min_resolved'] = float(res.P1.min())
N['dense_b_vs_8t_resolved'] = [int((res.best_b_8t > 1).sum()), len(res)]
N['dense_b_1t_range'] = [float(res.best_b_1t.min()), float(res.best_b_1t.max())]
N['dense_b_8t_range'] = [float(res.best_b_8t.min()), float(res.best_b_8t.max())]
dense = []
fc = pd.read_csv(OCT / 'fimi_lite_cost_stats.csv')
for d in ['chess', 'connect', 'mushroom', 'accidents', 'pumsb']:
    q = fx[fx.dataset == d]
    dense.append(f"{d} & {q.density.iloc[0]*100:.1f} & {rng(q.P1, 1)} & {rng(q['classic/scatter 1T'], 1)} & {rng(q['classic/scatter 8T'], 1)} & {rng(q.best_b_1t)} & {rng(q.best_b_8t)} \\\\")
rows('tab_dense_rows.tex', dense)


In [10]:
# ---- all datasets: real above the rule, synthetic below --------------------------------------------
SYNTH = st.index.str.match(r'T\d+\.I\d+\.')
fc1 = fc[np.isclose(fc.min_support, 1e-3)].set_index('dataset')
DMAX = {'chess': 37, 'connect': 43, 'mushroom': 23, 'pumsb': 74, 'accidents': 51}      # from the parquets in data/generated
real =[(name(d), r.rows, f'{name(d)} & {integer(r.D)} & {integer(r.N)} & {integer(r.F1)} & {r.d_avg:.2f} & {integer(r.d_max)} & {r.d_cv:.2f} & {integer(r.rows)} \\\\')
        for d, r in st[~SYNTH].iterrows()]
for d, r in fc1.iterrows():
    n = round(r.D * r.d_avg)
    real.append((d, n, f'{d}$^{{\\dagger}}$ & {integer(r.D)} & {integer(r.N)} & {integer(r.F1)} & {r.d_avg:.2f} & {integer(DMAX[d])} & {r.d_cv:.2f} & {integer(n)} \\\\'))
syn = [f'{d} (\\textbf{{{name(d)}}}) & {integer(r.D)} & {integer(r.N)} & {integer(r.F1)} & {r.d_avg:.2f} & {integer(r.d_max)} & {r.d_cv:.2f} & {integer(r.rows)} \\\\'
       for d, r in st[SYNTH].sort_values('rows').iterrows()]
rows('tab_datasets_rows.tex', [x[2] for x in sorted(real, key=lambda x: x[1])] + ['\\midrule'] + syn)

fl = pd.read_csv(OCT / 'fimi_lite.csv')
fl = fl[(~fl.warmup) & (fl.status == 'ok')]
fm = fl.groupby(['dataset', 'method', 'k', 'min_support']).wall_s.median().unstack('method').reset_index()
f3 = fm[fm.k == 3].copy()
f3['best'] = f3[['borgelt_eclat', 'borgelt_fpgrowth', 'lcm']].min(axis=1)
N['dense_k3_cells'] = len(f3)
N['dense_k3_8t_wins'] = int((f3.best > f3.hyb).sum())
f3r = f3[f3.dataset == 'connect'].set_index('min_support')
N['connect_k3_low'] = dict(hyb=float(f3r.loc[.001, 'hyb']), fp=float(f3r.loc[.001, 'borgelt_fpgrowth']), eclat=float(f3r.loc[.001, 'borgelt_eclat']))


In [11]:
# ---- scatter-seeded Eclat on the grid and the dense datasets ---------------------------------------
sall = pd.concat([pd.read_csv(NOV / f'{f}.csv') for f in ['seeded_grid', 'seeded_gaps_5e4', 'seeded_gaps_1e4', 'seeded_dense']])
sall['s'] = sall.min_support.round(7)
sok = sall[(sall.status == 'ok') & (~sall.warmup)]
assert (sok.groupby(['experiment', 'dataset', 'method', 'k', 's']).size() == 3).all()
assert sok.groupby(['dataset', 'k', 's']).n_itemsets.nunique().max() == 1
SM = sok.groupby(['dataset', 'k', 's', 'method']).agg(wall=('wall_s', 'median'), rss=('peak_rss_kb', 'median'), n=('n_itemsets', 'median'))
swall = SM.wall.unstack('method')
scount = SM.n.groupby(level=[0, 1, 2]).first()
sstat = sall[sall.warmup & (sall.method == 'hyb_seeded')].set_index(['dataset', 'k', 's']).status
spread = sok.groupby(['experiment', 'dataset', 'method', 'k', 's']).wall_s.agg(lambda x: x.max() / x.min()).groupby('experiment').median()
N['seeded_campaign'] = dict(completed=int(((sall.status == 'ok') & sall.warmup).sum()), measured=len(sok), failed=sall[sall.status.isin(['oom', 'timeout'])].status.value_counts().to_dict(),
                            skipped=int((sall.status == 'skipped').sum()), spread=[float(spread.min()), float(spread.max())])
m_1 = sall[np.isclose(sall.min_support, 1e-4) & (sall.method == 'hyb_seeded') & sall.warmup & (sall.experiment != 'seeded_dense')]
N['seeded_1e4_status'] = dict(attempted=len(m_1), by_status=m_1.status.value_counts().to_dict())
mc = ok[ok.method != 'efficient_apriori'].assign(s=lambda x: x.min_support.round(7)).groupby(['dataset', 'k', 's']).n_itemsets_median.first().rename('ref').reset_index()
mc = mc.merge(scount.rename('n').reset_index(), on=['dataset', 'k', 's'])
assert (mc.ref == mc.n).all()
N['seeded_count_cells_grid'] = len(mc)
sd = sok[sok.experiment == 'seeded_dense'].groupby(['dataset', 's', 'method']).agg(wall=('wall_s', 'median'), n=('n_itemsets', 'median')).reset_index()
dn3 = sd.pivot_table(index=['dataset', 's'], columns='method', values='wall').reset_index()
fl3 = fl[fl.k == 3].assign(s=lambda x: x.min_support.round(7))
fw = fl3.groupby(['dataset', 's', 'method']).wall_s.median().unstack('method').reset_index()
fn = fl3[fl3.method == 'hyb'].groupby(['dataset', 's']).n_itemsets.median().rename('nref').reset_index()
dx = dn3.merge(fw, on=['dataset', 's']).merge(fn, on=['dataset', 's']).merge(sd[sd.method == 'hyb_seeded'][['dataset', 's', 'n']], on=['dataset', 's'])
assert (dx.nref == dx.n).all()
dx['best'] = dx[['borgelt_eclat', 'borgelt_fpgrowth', 'lcm']].min(axis=1)
dx['r8'], dx['r_1'], dx['ga8'], dx['ga1'] = dx.best / dx.hyb_seeded, dx.best / dx.hyb_seeded_1t, dx.hyb / dx.hyb_seeded, dx.hyb_1t / dx.hyb_seeded_1t
N['seeded_dense'] = dict(cells=len(sd) // 2, compared=len(dx), wins_8t=int((dx.r8 > 1).sum()), wins_1t=int((dx.r_1 > 1).sum()), anchor_wins=int((dx.best > dx.hyb).sum()),
                         wins_by_dataset=dx[dx.r8 > 1].groupby('dataset').size().to_dict(), ratio_by_dataset=dx.groupby('dataset').r8.agg(['min', 'max']).round(2).to_dict('index'),
                         vs_anchor_8t=[float(dx.ga8.min()), float(dx.ga8.median()), float(dx.ga8.max())], vs_anchor_1t=[float(dx.ga1.min()), float(dx.ga1.median()), float(dx.ga1.max())],
                         pumsb_accidents_low=sd[(sd.s == .001) & sd.dataset.isin(['pumsb', 'accidents'])][['dataset', 'method', 'wall']].round(2).to_dict('records'))


In [12]:
# ---- higher order: lowest attempted support per (dataset, k) ---------------------------------------
fa = cov[cov.method == 'fastapriori_fast']
lowest = fa.groupby(['dataset', 'k']).min_support.min().rename('s').reset_index()
lowest = lowest.merge(fa, left_on=['dataset', 'k', 's'], right_on=['dataset', 'k', 'min_support'])
wall = ok.set_index(['dataset', 'k', 'min_support', 'method']).wall_s_median
items = ok[ok.method == 'fastapriori_fast'].set_index(['dataset', 'k', 'min_support']).n_itemsets_median
rec = []
for r in lowest.itertuples():
    e = [wall.get((r.dataset, r.k, r.s, m), np.nan) for m in EXT]
    e = np.nanmin(e) if np.isfinite(e).any() else np.nan
    f = wall.get((r.dataset, r.k, r.s, 'fastapriori_fast'), np.nan)
    rec.append(dict(dataset=r.dataset, k=r.k, s=r.s, status=r.probe_status, ext=e, fast=f, ratio=e / f,
                    n=items.get((r.dataset, r.k, r.s), np.nan)))
pl = pd.DataFrame(rec)
key = list(zip(pl.dataset, pl.k, pl.s.round(7)))
pl['s8'] = [swall.hyb_seeded.get(c, np.nan) for c in key]
pl['s1'] = [swall.hyb_seeded_1t.get(c, np.nan) for c in key]
pl['sn'] = [scount.get(c, np.nan) for c in key]
pl['sstat'] = [sstat.get(c, 'none') for c in key]
pl['r8'], pl['r_1'] = pl.ext / pl.s8, pl.ext / pl.s1
N['perlevel_k3'] = dict(attempted=int((pl.k == 3).sum()), wins=int(((pl.k == 3) & (pl.ratio > 1)).sum()))
def ratio_cell(v, base):
    if v == 0:                                             # comparator at Borgelt's 10 ms clock
        return f'$<${.01 / base:.2f}'
    return f'\\textbf{{{v:.2f}}}' if v > 1 else f'{v:.2f}'
tab = []
for d in sorted(pl.dataset.unique(), key=lambda x: name(x).lower()):
    if (pl[(pl.dataset == d)].k > 2).sum() == 0:
        continue
    q = pl[pl.dataset == d].set_index('k')
    cells = []
    for k in range(3, 10):
        if k not in q.index:
            cells.append('---')
        elif q.loc[k, 'status'] != 'ok':
            cells.append('\\textsc{oom}' if q.loc[k, 'status'] == 'oom' else '\\textsc{to}')
        else:
            cells.append(ratio_cell(q.loc[k, 'ratio'], q.loc[k, 'fast']))
    for k in range(3, 10):
        if k not in q.index:
            cells.append('---')
        elif q.loc[k, 'sstat'] == 'oom':
            cells.append('\\textsc{oom}')
        elif q.loc[k, 'sstat'] != 'ok':
            cells.append('---')
        else:
            cells.append(ratio_cell(q.loc[k, 'r8'], q.loc[k, 's8']))
    mark = '' if np.isclose(q[q.index >= 3].s.max(), 1e-4) else '$^{*}$'
    tab.append(f'{name(d)}{mark} & ' + ' & '.join(cells[:1] + cells[7:]) + ' \\\\')      # ANCHOR at k=3, then SEEDED
rows('tab_k3ratio_rows.tex', tab)
rssA = ok[ok.method == 'fastapriori_fast'].assign(s=lambda x: x.min_support.round(7)).set_index(['dataset', 'k', 's']).peak_rss_kb_median
pl['rA'] = [rssA.get(c, np.nan) for c in key]
pl['rS'] = [SM.rss.get(c + ('hyb_seeded',), np.nan) for c in key]
h = pl[(pl.k >= 3) & pl.ext.notna() & pl.s8.notna()]
both = h[h.fast.notna()]
ga = both.fast / both.s8
wins = lambda q: dict(cells=len(q), seeded_8t=int((q.r8 > 1).sum()), seeded_1t=int((q.r_1 > 1).sum()), anchor_8t=int((q.ratio > 1).sum()), median_8t=round(float(q.r8.median()), 3),
                      median_1t=round(float(q.r_1.median()), 3), median_anchor=round(float(q.ratio.median()), 3))
N['seeded_table'] = dict(all=wins(h), nonempty=wins(h[h.sn > 0]), thousand=wins(h[h.sn >= 1000]), s_1e4=wins(h[np.isclose(h.s, 1e-4)]),
                         vs_anchor=dict(cells=len(both), faster=int((ga > 1).sum()), median=float(ga.median()), min=float(ga.min()), max=float(ga.max()),
                                        slower=both[ga < 1][['dataset', 'k', 'fast', 's8']].round(3).to_dict('records')),
                         rss_vs_anchor=[float((both.rS / both.rA).min()), float((both.rS / both.rA).median()), float((both.rS / both.rA).max())],
                         wins_by_dataset=h[h.r8 > 1].groupby('dataset').size().to_dict(), anchor_wins_by_dataset=h[h.ratio > 1].groupby('dataset').size().to_dict(),
                         ratio_range={name(d): [round(float(g.r8.min()), 2), round(float(g.r8.max()), 2)] for d, g in h.groupby('dataset')},
                         wins_1t_by_dataset=h[h.r_1 > 1].groupby('dataset').size().to_dict(), best_ext=None)
af = pl[(pl.k >= 3) & (pl.status != 'ok')]
N['seeded_on_anchor_failures'] = dict(cells=len(af), statuses=af.sstat.value_counts().to_dict(), completed=af[af.s8.notna()][['dataset', 'k', 's', 's8', 'ext', 'sn']].round(5).to_dict('records'))
N['seeded_gaps_1e4'] = SM.wall.unstack('method').loc[lambda z: z.index.get_level_values('dataset').isin(['BMS-WebView-1', 'Kosarak', 'Online Retail', 'T12.I4.D5k.N1k.L1k']) & np.isclose(z.index.get_level_values('s'), 1e-4) & z.borgelt_eclat.notna()].round(2).reset_index().to_dict('records')

allhi = []
for (d, k, s), grp in ok[ok.k >= 3].groupby(['dataset', 'k', 'min_support']):
    v = grp.set_index('method').wall_s_median
    if 'fastapriori_fast' in v.index and v.reindex(EXT).notna().any():
        allhi.append(dict(dataset=d, k=k, s=s, win=v.fastapriori_fast < v.reindex(EXT).min(),
                          n=grp.set_index('method').n_itemsets_median.get('fastapriori_fast', np.nan)))
allhi = pd.DataFrame(allhi)
wn = allhi[allhi.win]
N['hi_wins'] = dict(wins=len(wn), cells=len(allhi), empty=int((wn.n == 0).sum()), under_1000=int(((wn.n > 0) & (wn.n < 1000)).sum()),
                    substantial=wn[wn.n >= 1000][['dataset', 'k', 's', 'n']].to_dict('records'))
counts = ok[(ok.method == 'fastapriori_fast') & (ok.k.isin([2, 3]))]
out = []
for support in sorted(counts.min_support.unique()):
    line = [f'${support:g}$']
    for k in [2, 3]:
        q = counts[(counts.k == k) & np.isclose(counts.min_support, support)]
        line += [str(len(q)), integer(q.n_itemsets_median.median()), integer(q.n_itemsets_median.max())]
    out.append(' & '.join(line) + ' \\\\')
rows('tab_output_counts_rows.tex', out)

r_3 = raw[(raw.dataset == 'Groceries') & (raw.k == 6) & np.isclose(raw.min_support, 1e-4)].set_index('method')
N['groceries_k6'] = dict(hyb_s=float(r_3.wall_s_median['fastapriori_fast']), hyb_rss_gb=float(r_3.peak_rss_kb_median['fastapriori_fast'] / 1e6),
                         eclat_s=float(r_3.wall_s_median['borgelt_eclat']), eclat_rss_mb=float(r_3.peak_rss_kb_median['borgelt_eclat'] / 1e3))


In [13]:
# ---- October sweeps ------------------------------------------------------------------------------
def med(file):
    d = pd.read_csv(OCT / f'{file}.csv')
    assert (d.status == 'ok').all()
    d = d[~d.warmup]
    m = d.groupby(['dataset', 'method', 'k']).agg(wall=('wall_s', 'median'), rss=('peak_rss_kb', 'median'), n=('n_itemsets', 'median')).reset_index()
    m['x'] = m.dataset.str.extract(r'(\d+)$').astype(int)
    m['rss'] = m.rss / 2**20
    assert d.groupby(['dataset', 'k']).n_itemsets.nunique().max() == 1
    return m
sw = {'D': med('scaling_D'), 'T': med('scaling_T'), 'dmax': med('scaling_dmax')}
ALL5 = ['borgelt_apriori', 'borgelt_eclat', 'pyfim_eclat', 'pyfim_fpgrowth', 'lcm']
seed = []
for key, m in sw.items():
    for k in [3, 4]:
        p = m[m.k == k].pivot_table(index='x', columns='method', values='wall')
        e = p[ALL5].min(axis=1)
        seed.append(pd.DataFrame(dict(sweep=key, k=k, x=p.index, ext=e.values, s8=p.hyb_seeded.values, s1=p.hyb_seeded_1t.values,
                                      a8=p.hyb.values, a1=p.hyb_1t.values)))
seed = pd.concat(seed)
seed['r8'], seed['r_1'] = seed.ext / seed.s8, seed.ext / seed.s1
seed['g8'], seed['g1'] = seed.a8 / seed.s8, seed.a1 / seed.s1
N['seeded_sweeps'] = dict(cells=len(seed), wins_8t=int((seed.r8 > 1).sum()), wins_1t=int((seed.r_1 > 1).sum()), ratio_8t=[float(seed.r8.min()), float(seed.r8.max())],
                          ratio_8t_median=float(seed.r8.median()), ratio_1t=[float(seed.r_1.min()), float(seed.r_1.max())], ratio_1t_median=float(seed.r_1.median()),
                          vs_anchor_8t=[float(seed.g8.min()), float(seed.g8.max())], vs_anchor_1t=[float(seed.g1.min()), float(seed.g1.max())],
                          anchor_8t_wins=int((seed.ext > seed.a8).sum()), anchor_1t_wins=int((seed.ext > seed.a1).sum()),
                          losses_1t=seed[seed.r_1 <= 1][['sweep', 'k', 'x', 'r_1']].to_dict('records'))
slopes = {}
for meth in ['hyb', 'hyb_1t', 'hyb_seeded', 'hyb_seeded_1t', 'borgelt_eclat', 'borgelt_apriori', 'pyfim_fpgrowth', 'lcm']:
    for key in ['D', 'T']:
        for k in [2, 3]:
            q = sw[key][(sw[key].method == meth) & (sw[key].k == k)].sort_values('x')
            slopes[f'{meth}/{key}/k{k}'] = float(np.polyfit(np.log(q.x), np.log(q.wall), 1)[0])
N['slopes'] = slopes
for key in sw:
    for k in [2, 3]:
        p = sw[key][sw[key].k == k].pivot_table(index='x', columns='method', values='wall')
        N[f'sweep_{key}_k{k}'] = {str(c): [round(float(v), 2) for v in p[c]] for c in p.columns}
        N[f'sweep_{key}_x'] = [int(v) for v in p.index]
p = sw['D'][sw['D'].k == 3].pivot_table(index='x', columns='method', values='rss')
N['rss_D5M_k3'] = {c: round(float(p.loc[5000000, c]), 2) for c in p.columns}
N['itemsets_D'] = {k: sw['D'][(sw['D'].k == k) & (sw['D'].method == 'hyb')].sort_values('x').n.astype(int).tolist() for k in [2, 3, 4]}
N['itemsets_T'] = {k: sw['T'][(sw['T'].k == k) & (sw['T'].method == 'hyb')].sort_values('x').n.astype(int).tolist() for k in [2, 3, 4]}


In [14]:
# ---- memory and CLI boundary ---------------------------------------------------------------------
mb = pd.read_csv(OCT / 'memory_boundary.csv')
oom = mb[mb.status == 'oom'].iloc[0]
N['kosarak_oom_rss_gb'] = float(oom.peak_rss_kb / 1e6)
mb = mb[(~mb.warmup) & (mb.status == 'ok')]
mm = mb.groupby(['dataset', 'method', 'k']).agg(p=('process_wall_s', 'median'), rss=('peak_rss_kb', 'median'), read=('borgelt_read_s', 'median'), n=('n_itemsets', 'median')).reset_index()
mm['rss'] = mm.rss / 2**20
order = ['Chainstore', 'Kosarak', 'Instacart', 'T20.I4.D1000k.N50k.L50k', 'T30.I8.D2000k.N10k.L10k', 'T25.I8.D5000k.N40k.L40k']
terms = pd.read_csv(OCT / 'memory_terms.csv').set_index('dataset')
pk = mm.pivot_table(index=['dataset', 'k'], columns='method', values='p')
rk = mm.pivot_table(index=['dataset', 'k'], columns='method', values='rss')
rd = mm.pivot_table(index=['dataset', 'k'], columns='method', values='read')
out, seedrows, sr = [], [], []
for d in order:
    a, b, c = pk.loc[(d, 3)], rk.loc[(d, 3)], rk.loc[(d, 2)]
    e3 = a.borgelt_eclat - rd.loc[(d, 3)].borgelt_eclat
    anch = '\\textsc{oom}' if pd.isna(a.hybcli_anchor) else f'{a.hybcli_anchor:.1f} / {b.hybcli_anchor:.2f}'
    best3 = min(a.borgelt_eclat, a.borgelt_fpgrowth, a.lcm)
    sr.append(dict(d=d, seeded=a.hybcli_seeded, best=best3, eclat=a.borgelt_eclat, excl=e3, rss=b.hybcli_seeded, anchor=a.hybcli_anchor, anchor_rss=b.hybcli_anchor))
    seedrows.append(f'{name(d)} & {anch} & \\textbf{{{a.hybcli_seeded:.1f}}} / {b.hybcli_seeded:.2f} & {a.borgelt_eclat:.1f} / {b.borgelt_eclat:.2f} & {a.borgelt_fpgrowth:.1f} / {b.borgelt_fpgrowth:.2f} & {a.lcm:.1f} / {b.lcm:.2f} \\\\')
    t = terms.loc[d]
    net = c.hybcli_pairs_v0 - c.hybcli_load
    out.append(f'{name(d)} & {c.hybcli_load:.2f} & {c.hybcli_pairs_v0:.2f} & {net:.2f} & {t.index_MB/1024:.2f} & {c.borgelt_apriori:.2f} & {t.triangular_MB/1024:.2f} & {c.borgelt_eclat:.2f} & {c.lcm:.2f} \\\\')
rows('tab_memory_rows.tex', out)
rows('tab_seeded_rows.tex', seedrows)
sr = pd.DataFrame(sr)
sr['r'], sr['re'] = sr.best / sr.seeded, sr.excl / sr.seeded
N['seeded_cli'] = dict(wins=int((sr.r > 1).sum()), n=len(sr), ratio=[float(sr.r.min()), float(sr.r.max())], ratio_vs_eclat_excl_read=[float(sr.re.min()), float(sr.re.max())],
                       detail=sr.round(3).to_dict('records'))
c2 = pk.xs(2, level='k')
N['cli_k2'] = dict(v0=c2.hybcli_pairs_v0.round(2).to_dict(), v2=c2.hybcli_pairs_v2.round(2).to_dict(), v0_1t=c2.hybcli_pairs_v0_1t.round(2).to_dict(), v2_1t=c2.hybcli_pairs_v2_1t.round(2).to_dict())
ext3 = c2[['borgelt_apriori', 'borgelt_eclat', 'borgelt_fpgrowth', 'lcm']].min(axis=1)
N['cli_k2_wins'] = dict(v0_8t=int((ext3 > c2.hybcli_pairs_v0).sum()), v2_8t=int((ext3 > c2.hybcli_pairs_v2).sum()), v2_1t=int((ext3 > c2.hybcli_pairs_v2_1t).sum()), n=len(c2))
N['v2_gain_1t'] = [float((c2.hybcli_pairs_v0_1t / c2.hybcli_pairs_v2_1t).min()), float((c2.hybcli_pairs_v0_1t / c2.hybcli_pairs_v2_1t).max())]


In [15]:
# ---- phase profile -------------------------------------------------------------------------------
pt = pd.read_csv(OCT / 'phaseprof_table.csv')
pp = pd.read_csv(OCT / 'phaseprof.csv')
pp = pp[~pp.warmup]
prod = pp[pp.method.isin(['hyb', 'hyb_1t', 'borgelt_eclat'])].groupby(['dataset', 'method', 'k']).wall_s.median().unstack('method').reset_index()
prod['w'] = prod.dataset.map(name)
prod['r8'], prod['r_1'] = prod.borgelt_eclat / prod.hyb, prod.borgelt_eclat / prod.hyb_1t
N['phase_production'] = prod[['w', 'k', 'hyb', 'hyb_1t', 'borgelt_eclat', 'r8', 'r_1']].round(3).to_dict('records')
pt['setup'] = pt.input_encoding + pt['index']
pt['tail'] = pt.output + pt.other
for c in ['setup', 'pairs', 'ext_setup', 'ext_k3', 'ext_k4', 'tail']:
    pt[c + '_sh'] = pt[c] / pt.inner_wall_s
N['phase_k3'] = pt[pt.k == 3][['workload', 'threads', 'n_frequent_k', 'inner_wall_s', 'setup', 'input_encoding', 'index', 'pairs', 'ext_setup', 'ext_k3', 'setup_sh', 'pairs_sh', 'ext_setup_sh', 'ext_k3_sh']].round(3).to_dict('records')
N['phase_k4'] = pt[pt.k == 4][['workload', 'threads', 'n_frequent_k', 'pairs_sh', 'ext_setup_sh', 'ext_k3_sh', 'ext_k4_sh', 'setup_sh']].round(3).to_dict('records')
ov = pd.read_csv(OCT / 'phaseprof_overhead.csv')
N['overhead'] = dict(mt=[float(ov.mt_ratio.min()), float(ov.mt_ratio.max())], t1=[float(ov['1t_ratio'].min()), float(ov['1t_ratio'].max())], flagged=int(ov.flagged.sum()), n=len(ov))
wk = pd.read_csv(OCT / 'phaseprof_work.csv')
q = wk[(wk.threads == 1) & (wk.k == 3)][['workload', 'k3_extend_ns_per_tid', 'k3_project_ns_per_txn', 'k3_anchor_tids', 'k3_n_anchors', 'k3_n_frequent']]
N['phase_work_1t'] = q.round(1).to_dict('records')


In [16]:
# ---- machine-readable checks on the May derived tables --------------------------------------------
for f, lab in [('ablation', 'tab:ablation')]:
    t = (DER / 'part_c' / f'{f}.tex').read_text(encoding='utf-8')
    cap = {'ablation': r'\\caption{Cumulative ablation over successive builds: v0 sorted lists, v1 adds Roaring lists, v2 adds memoization and v3, the production build, adds the adaptive representation. A1 is v0/v1, A2a and A2b are v1/v2, A3 is v2/v3 and A4 is one thread over eight on v3, so these are not independent single-component effects. Above 1, the later build is faster. $s=10^{-3}$ for Kosarak, $10^{-4}$ elsewhere; A1, A2a, A3 at $k=3$, A2b at $k=4$, A4 at $k=2$.}'}[f]
    t = re.sub(r'\\caption\{.*?\}\\label', cap + r'\\label', t, count=1, flags=re.S)
    (GEN / f'{f}.tex').write_text(t, encoding='utf-8')


In [17]:
# ---- large workloads, resolved by level --------------------------------------------------------
BIG = ['T20.I4.D1000k.N50k.L50k', 'T30.I8.D2000k.N10k.L10k', 'T25.I8.D5000k.N40k.L40k']
SEEDROWS = [('\\seeded{} (8T)', 'hyb_seeded'), ('\\seeded{} (1T)', 'hyb_seeded_1t')]
ROWS = [('\\fast{} (8T)', 'fastapriori_fast'), ('\\fastoneT{}', 'fastapriori_fast_1t')] + SEEDROWS + [('\\classic{}', 'fastapriori_classic'),
        ('\\pyfim{} Apriori', 'pyfim_apriori'), ('\\pyfim{} Eclat', 'pyfim_eclat'), ('\\pyfim{} FP-Growth', 'pyfim_fpgrowth'),
        ('\\borgelt{} Apriori', 'borgelt_apriori'), ('\\borgelt{} Eclat', 'borgelt_eclat'), ('\\borgelt{} FP-Growth', 'borgelt_fpgrowth')]
lcm = pd.read_csv(MAY / 'lcm_declat_results_20260523_055523.csv')
lcm = lcm[(lcm.status == 'ok') & np.isclose(lcm.min_support, 1e-4)].set_index(['dataset', 'k'])
may1e4 = lo.set_index(['dataset', 'k', 'method'])
def fmt(sec, gib, bold_t=False, bold_m=False):
    if pd.isna(sec):
        return '---'
    s = f'{sec:.0f}' if sec >= 100 else f'{sec:.1f}'
    g = f'{gib:.2f}' if gib < 1 else f'{gib:.1f}'
    return ('\\textbf{' + s + '}' if bold_t else s) + ' / ' + ('\\textbf{' + g + '}' if bold_m else g)
cells = {}
for d in BIG:
    for k in [2, 3]:
        v = {}
        for lab, m in ROWS:
            if m.startswith('hyb_seeded'):
                v[lab] = (float(SM.wall[(d, 3, 1e-4, m)]), float(SM.rss[(d, 3, 1e-4, m)]) / 2**20) if k == 3 else (np.nan, np.nan)
            elif (d, k, m) in may1e4.index:
                r = may1e4.loc[(d, k, m)]
                v[lab] = (float(r.wall_s_median), float(r.peak_rss_kb_median) / 2**20)
            else:
                v[lab] = (np.nan, np.nan)
        if (d, k) in lcm.index:
            v['\\lcm{} v5.3'] = (float(lcm.loc[(d, k)].wall_s), float(lcm.loc[(d, k)].peak_rss_mb) / 1024)
        else:
            v['\\lcm{} v5.3'] = (np.nan, np.nan)
        cells[(d, k)] = v
table = []
HIDE = {'\\pyfim{} Apriori', '\\pyfim{} FP-Growth', '\\borgelt{} FP-Growth'}      # never fastest or smallest in these columns
for lab in [x[0] for x in ROWS if x[0] not in HIDE] + ['\\lcm{} v5.3']:
    row = [lab]
    for d in BIG:
        for k in [2, 3]:
            v = cells[(d, k)]
            bt = min(x[0] for x in v.values() if not np.isnan(x[0]))
            bm = min(x[1] for x in v.values() if not np.isnan(x[1]))
            row.append(fmt(*v[lab], v[lab][0] == bt, v[lab][1] == bm))
    table.append(' & '.join(row) + ' \\\\')
rows('tab_bigdata_rows.tex', table)

saved = (best - pair.fastapriori_fast).sort_values()
N['k2_saved_largest'] = {name(k): round(float(v), 2) for k, v in saved.tail(6).items()}
N['classic_saved_T20'] = float(pair.loc['T20.I4.D1000k.N50k.L50k'].fastapriori_classic - pair.loc['T20.I4.D1000k.N50k.L50k'].fastapriori_fast)
pf = pair[['pyfim_apriori', 'pyfim_eclat', 'pyfim_fpgrowth']].min(axis=1)
bo = pair[['borgelt_apriori', 'borgelt_eclat', 'borgelt_fpgrowth']].min(axis=1)
N['k2_vs_pyfim'] = dict(wins_8t=int((pf > pair.fastapriori_fast).sum()), wins_1t=int((pf > pair.fastapriori_fast_1t).sum()), n=len(pf),
                        median_8t=float((pf / pair.fastapriori_fast).median()), max_8t=float((pf / pair.fastapriori_fast).max()))
N['k2_vs_borgelt'] = dict(wins_8t=int((bo > pair.fastapriori_fast).sum()), wins_1t=int((bo > pair.fastapriori_fast_1t).sum()), n=len(bo),
                          median_8t=float((bo / pair.fastapriori_fast).median()), median_1t=float((bo / pair.fastapriori_fast_1t).median()),
                          range_8t=[float((bo / pair.fastapriori_fast).min()), float((bo / pair.fastapriori_fast).max())])
big8 = pd.Series({name(d): bo[d] / pair.fastapriori_fast[d] for d in st.sort_values('rows').index[-6:]})
N['k2_vs_borgelt_large'] = big8.round(2).to_dict()
ea = pair.dropna(subset=['efficient_apriori'])
N['k2_vs_efficient_apriori'] = dict(n=len(ea), wins_8t=int((ea.efficient_apriori > ea.fastapriori_fast).sum()),
                                    ratio=[float((ea.efficient_apriori / ea.fastapriori_fast).min()), float((ea.efficient_apriori / ea.fastapriori_fast).max())],
                                    median=float((ea.efficient_apriori / ea.fastapriori_fast).median()))
q = seed[seed.sweep == 'D']
N['seeded_saved_D5M'] = float((q[(q.k == 3) & (q.x == 5000000)].ext - q[(q.k == 3) & (q.x == 5000000)].s8).iloc[0])
b1 = pd.read_csv(OCT / 'a1_bounds.csv')
b1 = b1[~b1.warmup].groupby(['dataset', 'method', 'k']).agg(w=('wall_s', 'median'), n=('n_itemsets', 'median')).reset_index()
bb = b1[b1.method.str.endswith('_nobound')].assign(method=lambda x: x.method.str.replace('_nobound', '')).merge(b1, on=['dataset', 'method', 'k'], suffixes=('_u', '_b'))
bb['r'] = bb.w_u / bb.w_b
N['bounds'] = dict(n=len(bb), ratio=[float(bb.r.min()), float(bb.r.max())], median=float(bb.r.median()),
                   rows=bb[['dataset', 'method', 'k', 'w_b', 'w_u', 'r', 'n_b', 'n_u']].round(2).to_dict('records'))


In [18]:
# ---- diagnostic numbers --------------------------------------------------------------------------
ml_sum = pd.read_csv(GEN / 'ml_summary.csv').set_index('task')
ml_sweep = pd.read_csv(GEN / 'ml_threshold_sweep.csv')
ml_imp = pd.read_csv(GEN / 'ml_importance.csv').set_index('feature')
N['ml'] = ml_sum.round(3).to_dict('index')
N['ml_top5'] = {t: ml_imp[t].sort_values(ascending=False).head(5).round(3).to_dict() for t in ml_imp.columns}
N['ml_corr'] = pd.read_csv(GEN / 'ml_feature_correlation.csv', index_col=0).round(3).to_dict()


In [19]:
# ---- figures -------------------------------------------------------------------------------------
from matplotlib.ticker import NullLocator, FixedLocator, NullFormatter, FuncFormatter
def sci(v):
    e = int(np.floor(np.log10(v) + 1e-9))
    m = v / 10.0 ** e
    return f'$10^{{{e}}}$' if abs(m - 1) < 1e-6 else f'${m:g}\\cdot10^{{{e}}}$'
def log_axis(ax, which, lo, hi, ticks, fmt=sci):
    """Log axis ending exactly on its first and last labelled values, with no minor marks."""
    axis = ax.xaxis if which == 'x' else ax.yaxis
    (ax.set_xscale if which == 'x' else ax.set_yscale)('log')
    (ax.set_xlim if which == 'x' else ax.set_ylim)(lo, hi)
    axis.set_major_locator(FixedLocator(ticks))
    axis.set_major_formatter(FuncFormatter(lambda v, _: fmt(v)))
    axis.set_minor_locator(NullLocator())
    axis.set_minor_formatter(NullFormatter())
def plain(v): return f'{v:g}'
def free(ax):
    for a in list(ax.lines) + [c for c in ax.collections if c.get_offsets().size]:
        a.set_clip_on(False)
        a.set_in_layout(False)
def decade_ticks(lo, hi):
    a, b = int(np.floor(np.log10(lo))), int(np.ceil(np.log10(hi)))
    e = list(range(a, b + 1))
    if len(e) > 4:
        e = sorted({a, (a + b) // 2, b})
    return 10.0 ** a, 10.0 ** b, [10.0 ** x for x in e]
def square(ax):
    ax.set_box_aspect(1)
    ax.spines[['top', 'right']].set_visible(True)
def place(fig, axes, left, top, size, gap_x, gap_y=0):
    """Square panels at fixed positions in inches, with no spare space around them."""
    W, H = fig.get_size_inches()
    for (r, c), ax in np.ndenumerate(np.atleast_2d(axes)):
        ax.set_position([(left + c * (size + gap_x)) / W, 1 - (top + (r + 1) * size + r * gap_y) / H, size / W, size / H])
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.size': 7.5, 'font.family': 'serif', 'pdf.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.linewidth': .6,
                     'xtick.major.size': 0, 'ytick.major.size': 0, 'xtick.minor.size': 0, 'ytick.minor.size': 0, 'axes.titlesize': 8})
ORANGE, CLASSIC, RED, PURPLE, BROWN, GREY, PINK, TEAL = '#FF5C00', '#2980B9', '#d62728', '#9467bd', '#8c564b', '#7f7f7f', '#e377c2', '#00A087'
STY = {'hyb_seeded': ('SEEDED 8T', TEAL, 'o', '-', True), 'hyb_seeded_1t': ('SEEDED 1T', TEAL, 'o', '--', False),
       'hyb': ('SCATTER / ANCHOR 8T', ORANGE, 's', '-', True), 'hyb_1t': ('SCATTER / ANCHOR 1T', ORANGE, 's', '--', False),
       'borgelt_eclat': ('Borgelt Eclat', PINK, '^', '-', True), 'pyfim_eclat': ('PyFIM Eclat', PURPLE, 's', '-', True),
       'pyfim_fpgrowth': ('PyFIM FP-Growth', PURPLE, 'v', '--', False),
       'borgelt_apriori': ('Borgelt Apriori', BROWN, 'D', '-', True), 'lcm': ('LCM', GREY, 'P', '-', True)}
best_ext = pd.concat([sw_tab[sw_tab.k.isin([2, 3])].pivot_table(index=['x', 'k'], columns='method', values='wall')[ALL5].idxmin(axis=1) for sw_tab in sw.values()])
N['pyfim_eclat_best_cells'] = [int((best_ext == 'pyfim_eclat').sum()), len(best_ext)]
fig, axes = plt.subplots(2, 3, figsize=(7.1, 3.62))
LAB = {'D': ('Transactions $D$', (1e5, 5e6), [1e5, 1e6, 5e6], sci), 'T': ('Mean width $T$', (5, 50), [5, 10, 20, 50], plain),
       'dmax': (r'Injected tail width $d_{\max}$', (50, 2500), [50, 500, 2500], plain)}
for c, key in enumerate(['D', 'T', 'dmax']):
    for r, k in enumerate([2, 3]):
        ax = axes[r, c]
        q = sw[key][sw[key].k == k]
        for meth, (lab, col, mk, ls, fill) in STY.items():
            if k == 2 and meth.startswith('hyb_seeded'):
                continue
            s = q[q.method == meth].sort_values('x')
            ax.plot(s.x, s.wall, color=col, marker=mk, ls=ls, lw=1.1, ms=3.2, mfc=col if fill else '#ffffff', mec=col, mew=.7, label=lab, clip_on=False, zorder=3)
        y0, y1, ticks = decade_ticks(q[q.method.isin(STY)].wall.min(), q[q.method.isin(STY)].wall.max())
        log_axis(ax, 'y', y0, y1, ticks)
        log_axis(ax, 'x', *LAB[key][1], LAB[key][2], LAB[key][3])
        free(ax)
        square(ax)
        ax.grid(True, which='major', color=GRID, lw=.5)
        ax.set_axisbelow(True)
        ax.set_title(f'$k={k}$', loc='center', fontsize=7, fontweight='bold', pad=3)
        if r == 1:
            ax.set_xlabel(LAB[key][0])
        if c == 0:
            ax.set_ylabel('Runtime (s)')
hd, lb = axes[1, 0].get_legend_handles_labels()
place(fig, axes, left=.5, top=.17, size=1.38, gap_x=.38, gap_y=.37)
fig.legend(hd, lb, loc='center left', bbox_to_anchor=(.772, .52), ncol=1, frameon=False, fontsize=6.5, labelspacing=.9)
fig.savefig(FIG / 'scaling.pdf')
plt.close(fig)

SEG = [('setup', 'Encoding and index', '#6250d6'), ('pairs', 'Pair counting', '#2a78d6'), ('ext_setup', 'Pair TID lists', '#1baf7a'),
       ('ext_k3', 'Extension to $k=3$', '#eb6834'), ('tail', 'Output and other', '#eda100')]
fig, axes = plt.subplots(1, 2, figsize=(7.1, 1.6), layout='constrained', sharey=True)
ordw = ['T10.D100k', 'T20.D1M', 'T25.D5M', 'Chainstore', 'Instacart', 'T30.D2M']
for ax, th, title in [(axes[0], 1, 'One thread'), (axes[1], 8, 'Eight threads')]:
    q = pt[(pt.k == 3) & (pt.threads == th)].set_index('workload').loc[ordw]
    left = np.zeros(len(q))
    total = prod[prod.k == 3].set_index('w').loc[ordw, 'hyb' if th == 8 else 'hyb_1t'].values
    eclat3 = prod[prod.k == 3].set_index('w').loc[ordw, 'borgelt_eclat'].values
    profiled = q[[c for c, _, _ in SEG]].sum(axis=1).values
    N[f'phase_bar_total_{th}t'] = dict(zip(ordw, np.round(total / eclat3, 3).tolist()))
    for c, lab, col in SEG:
        v = (q[c].values / profiled) * total / eclat3
        ax.barh(np.arange(len(q)), v, left=left, color=col, height=.68, edgecolor='#ffffff', lw=.5, label=lab)
        left += v
    ax.axvline(1, color=INK, ls='--', lw=.7)
    ax.set_title(title, loc='center')
    ax.set_xlabel('Runtime relative to Borgelt Eclat')
    ax.grid(True, axis='x', color=GRID, lw=.5)
    ax.set_axisbelow(True)
    ax.set_yticks(np.arange(len(q)), [f'{w} ({int(n):,})' for w, n in zip(q.index, q.n_frequent_k)])
    ax.invert_yaxis() if ax is axes[0] else None
    ax.tick_params(axis='y', length=0)
hd, lb = axes[0].get_legend_handles_labels()
fig.legend(hd, lb, loc='outside lower center', ncol=5, frameon=False, fontsize=7)
fig.savefig(FIG / 'phases.pdf')
plt.close(fig)


In [20]:
# ---- dataset landscape ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(3.35, 2.25), layout='constrained')
ax.scatter([d for d in (100e3, 250e3, 500e3, 1e6, 2.5e6, 5e6)] + [500e3] * 5, [20] * 6 + [5, 10, 20, 30, 50], s=13, marker='x', color=MUTED, lw=1, label='Sweeps', zorder=1, clip_on=False)
gr = st.assign(real=~st.index.str.startswith(('T', 'low', 'high')))
gr.loc[gr.index.str.contains('T15'), 'real'] = False
size = lambda r: 8 + 60 * np.sqrt(np.asarray(r, float) / 125e6)
for mask, lab, col, mk in [(gr.real, 'Real', '#2a78d6', 'o'), (~gr.real, 'Synthetic', '#eb6834', 's')]:
    q = gr[mask]
    ax.scatter(q.D, q.d_avg, s=size(q.rows), marker=mk, color=col, edgecolor='#ffffff', lw=.4, label=f'{lab} ({len(q)})', zorder=3, clip_on=False)
dn = fc.groupby('dataset').agg(D=('D', 'first'), d_avg=('d_avg', 'first'))
ax.scatter(dn.D, dn.d_avg, s=size(dn.D * dn.d_avg), marker='^', color='#1baf7a', edgecolor='#ffffff', lw=.4, label=f'Dense ({len(dn)})', zorder=3, clip_on=False)
PLACE = {'pumsb': (3, 2), 'connect': (3, 2), 'chess': (3, 2), 'accidents': (3, 2), 'mushroom': (3, -6)}
for d, (dx, dy) in PLACE.items():
    ax.annotate(d, (dn.D[d], dn.d_avg[d]), fontsize=5.5, color=MUTED, xytext=(dx, dy), textcoords='offset points')
for d, lab, (dx, dy) in [('Instacart', 'Instacart', (-10, 6)), ('Kosarak', 'Kosarak', (-27, -7)), ('Groceries', 'Groceries', (-12, -8)), ('BMS-WebView-1', 'BMS-1', (3, -3)),
                         ('T25.I8.D5000k.N40k.L40k', 'T25.D5M', (-14, 6)), ('T12.I4.D5k.N1k.L1k', 'T12.D5k', (3, -2))]:
    ax.annotate(lab, (st.D[d], st.d_avg[d]), fontsize=5.5, color=MUTED, xytext=(dx, dy), textcoords='offset points')
ax.set(xlabel='Transactions $D$', ylabel=r'Mean width $d_{\mathrm{avg}}$')
log_axis(ax, 'x', 1e3, 1e7, [1e3, 1e4, 1e5, 1e6, 1e7])
log_axis(ax, 'y', 1, 100, [1, 10, 100])
free(ax)
ax.spines[['top', 'right']].set_visible(True)
ax.grid(True, which='major', color=GRID, lw=.5)
ax.set_axisbelow(True)
hd, lb = ax.get_legend_handles_labels()
fig.legend(hd, lb, loc='outside lower center', ncol=4, frameon=False, fontsize=6.2, handletextpad=.1, columnspacing=1)
fig.savefig(FIG / 'landscape.pdf')
plt.close(fig)


In [21]:
# ---- pairwise runtime on every dataset -------------------------------------------------------------
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
PF = ['pyfim_apriori', 'pyfim_eclat', 'pyfim_fpgrowth']
BO = ['borgelt_apriori', 'borgelt_eclat', 'borgelt_fpgrowth']
LINE = {'fast': dict(color=ORANGE, marker='o', ls='-', lw=1.1, ms=2.8), 'one': dict(color=ORANGE, marker='D', ls='--', lw=.9, ms=2.4, mfc='#ffffff', mew=.5),
        'classic': dict(color=CLASSIC, marker='v', ls='-', lw=1, ms=2.8), 'ea': dict(color=RED, marker='^', ls='-', lw=1, ms=2.8),
        'pyfim': dict(color=PURPLE, marker='s', ls='-', lw=1, ms=2.5), 'borgelt': dict(color=BROWN, marker='', ls=':', lw=1.2),
        'lcm': dict(color=GREY, marker='P', ls='-', lw=.9, ms=2.8), 'seeded': dict(color=TEAL, marker='o', ls='-', lw=1.1, ms=2.8),
        'seeded1': dict(color=TEAL, marker='D', ls='--', lw=.9, ms=2.4, mfc='#ffffff', mew=.5)}
def decades(ax):
    y0, y1 = ax.get_ylim()
    lo, hi, ticks = decade_ticks(y0, y1)
    log_axis(ax, 'y', lo, hi, ticks)
wide = ok[ok.k == 2].pivot_table(index=['dataset', 'min_support'], columns='method', values='wall_s_median')
grid_panels = [d for d in st[~SYNTH].sort_values('rows').index] + [d for d in st[SYNTH].sort_values('rows').index]
n_real = int((~SYNTH).sum())
fig = plt.figure(figsize=(7.1, 8.05), layout='constrained')
fig.get_layout_engine().set(w_pad=.03, h_pad=.03, hspace=.02, wspace=.03)
gs = fig.add_gridspec(5, 4)
axes = np.array([fig.add_subplot(gs[i // 4, i % 4]) for i in range(len(grid_panels))])
N['fig2_borgelt_censored'] = {}
for ax, d in zip(axes.ravel(), grid_panels):
    square(ax)
    q = wide.loc[d].sort_index()
    band = q[PF].dropna(how='all')
    ax.fill_between(band.index, band.min(axis=1), band.max(axis=1), color=PURPLE, alpha=.16, lw=0)
    ax.plot(band.index, band.min(axis=1), **LINE['pyfim'])
    raw_b = q[BO].min(axis=1).dropna()
    bq = raw_b.replace(0, np.nan).dropna()
    ax.plot(bq.index, bq.values, **LINE['borgelt'])
    for col, sty in [('efficient_apriori', 'ea'), ('fastapriori_classic', 'classic'), ('fastapriori_fast_1t', 'one'), ('fastapriori_fast', 'fast')]:
        s = q[col].dropna()
        ax.plot(s.index, s.values, **LINE[sty])
    if (raw_b == 0).any():
        N['fig2_borgelt_censored'][name(d)] = int((raw_b == 0).sum())
    ax.set_yscale('log')
    decades(ax)
    log_axis(ax, 'x', 1e-4, 1e-2, [1e-4, 1e-3, 1e-2])
    free(ax)
    ax.grid(True, which='major', color=GRID, lw=.45)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', which='both', length=0, labelsize=6, pad=2)
    ax.set_title(name(d), fontsize=6.3, fontweight='bold', pad=2.5, loc='center', color=INK)
fig.supxlabel('Minimum support', fontsize=7.5)
fig.supylabel('Runtime (s)', fontsize=7.5)
lax = fig.add_subplot(gs[4, 2:])
lax.axis('off')
lax.legend(handles=[Line2D([], [], **LINE['fast']), Line2D([], [], **LINE['one']), Line2D([], [], **LINE['classic']), Line2D([], [], **LINE['ea']),
                    (Patch(color=PURPLE, alpha=.16, lw=0), Line2D([], [], **LINE['pyfim'])), Line2D([], [], **LINE['borgelt'])],
           labels=['HybridFIM scatter (8T)', 'HybridFIM scatter (1T)', 'classic (8T)', 'efficient-apriori', 'PyFIM best (band: variants)', 'Borgelt best'],
           loc='center', ncol=1, frameon=False, fontsize=7.5, handlelength=2.6, labelspacing=.9)
fig.savefig(FIG / 'k2_per_variant.pdf')
plt.close(fig)


In [22]:
# ---- runtime against k on three contrasting workloads ----------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(7.1, 1.92))
hi = lo.pivot_table(index=['dataset', 'k'], columns='method', values='wall_s_median')
for ax, d in zip(axes, ['T25.I8.D5000k.N40k.L40k', 'T30.I8.D2000k.N10k.L10k', 'Instacart']):
    q = hi.loc[d].sort_index()
    ax.fill_between(q.index, q[PF].min(axis=1), q[PF].max(axis=1), color=PURPLE, alpha=.16, lw=0)
    ax.plot(q.index, q[PF].min(axis=1), clip_on=False, **{**LINE['pyfim'], 'ms': 3})
    ax.plot(q.index, q[BO].min(axis=1), clip_on=False, **LINE['borgelt'])
    ax.plot(q.index, q.fastapriori_classic, clip_on=False, **{**LINE['classic'], 'ms': 3})
    l = lcm.loc[d].sort_index() if d in lcm.index.get_level_values(0) else None
    if l is not None:
        ax.plot(l.index, l.wall_s, clip_on=False, **{**LINE['lcm'], 'ms': 3})
    ax.plot(q.index, q.fastapriori_fast, clip_on=False, **{**LINE['fast'], 'ms': 3.2})
    sq = swall.xs((d, 1e-4), level=['dataset', 's']).sort_index()
    ax.plot(sq.index, sq.hyb_seeded_1t, clip_on=False, **{**LINE['seeded1'], 'ms': 2.8})
    ax.plot(sq.index, sq.hyb_seeded, clip_on=False, **{**LINE['seeded'], 'ms': 3.2})
    ax.set_yscale('log')
    ax.set_xticks(range(2, 10))
    ax.set_xlim(2, 9)
    decades(ax)
    free(ax)
    ax.set_xticks(range(2, 10))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.grid(True, which='major', color=GRID, lw=.5)
    ax.set_axisbelow(True)
    square(ax)
    ax.set_title(f'{name(d)}', loc='center', fontsize=7, fontweight='bold', pad=3)
    ax.set_xlabel('$k$')
axes[0].set_ylabel('Runtime (s)')
fig.legend(handles=[Line2D([], [], **{**LINE['seeded'], 'ms': 3.2}), Line2D([], [], **{**LINE['seeded1'], 'ms': 2.8}), Line2D([], [], **{**LINE['fast'], 'ms': 3.2}),
                    (Patch(color=PURPLE, alpha=.16, lw=0), Line2D([], [], **{**LINE['pyfim'], 'ms': 3})),
                    Line2D([], [], **LINE['borgelt']), Line2D([], [], **{**LINE['classic'], 'ms': 3}), Line2D([], [], **{**LINE['lcm'], 'ms': 3})],
           labels=['SEEDED (8T)', 'SEEDED (1T)', 'SCATTER / ANCHOR (8T)', 'PyFIM best (band: variants)', 'Borgelt best', 'classic (8T)', 'LCM'],
           loc='center left', bbox_to_anchor=(.765, .54), ncol=1, frameon=False, fontsize=6.5, handlelength=2.2, labelspacing=.9)
place(fig, axes, left=.48, top=.17, size=1.4, gap_x=.34)
fig.savefig(FIG / 'runtime_vs_k.pdf')
plt.close(fig)


In [23]:
# ---- threshold sweep and feature importance --------------------------------------------------------
with plt.rc_context({'font.family': 'sans-serif', 'font.size': 10, 'axes.spines.top': True, 'axes.spines.right': True, 'axes.edgecolor': 'black', 'axes.linewidth': .8,
                     'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'xtick.major.size': 3.5, 'ytick.major.size': 3.5, 'axes.titlesize': 11}):
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))
    ax = axes[0]
    for task, col, lab in [('pyfim', PURPLE, 'PyFIM'), ('classic', CLASSIC, 'classic')]:
        q = ml_sweep[ml_sweep.task == task]
        ax.plot(q['T'], q.mcc, ls='-', marker='o', lw=1.6, ms=5, color=col, alpha=.9, label=f'MCC ({lab})')
        ax.plot(q['T'], q.auroc, ls='--', marker='s', lw=1.6, ms=5, color=col, alpha=.9, label=f'AUROC ({lab})')
        star = int(ml_sum.loc[task, 'T_star'])
        ax.axvline(star, color=col, lw=1.9, ls=':', alpha=.9, label=f'T*={star}s ({lab})')
    ax.set_xlabel('Threshold T (seconds)', fontsize=10)
    ax.set_ylabel('Metric value', fontsize=10)
    ax.set_xlim(0, 10.5)
    ax.set_xticks(np.arange(0, 11, 1))
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_locator(FixedLocator([0, .2, .4, .6, .8, 1]))
    ax.set_title('Threshold sweep (random folds)', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=.3)
    ax.legend(loc='lower center', fontsize=9, ncol=2, frameon=True)
    for ax, task, col, lab in [(axes[1], 'pyfim', PURPLE, 'vs PyFIM best'), (axes[2], 'classic', CLASSIC, 'vs classic')]:
        q = ml_imp[task].sort_values()
        ax.barh(np.arange(len(q)), q.values, color=col, height=.6)
        ax.set_yticks(np.arange(len(q)), q.index, fontsize=10)
        ax.set_xlabel('XGBoost feature importance', fontsize=10)
        ax.set_title(f'Feature importance {lab}', fontsize=11, fontweight='bold')
        ax.grid(True, axis='x', alpha=.3)
    fig.tight_layout()
fig.savefig(FIG / 'ml_sweep_importance.pdf')
plt.close(fig)


In [24]:
# ---- regime maps -----------------------------------------------------------------------------------
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
cmap = LinearSegmentedColormap.from_list('hybrid', [(0, PURPLE), (.5, '#f5f5f5'), (.65, '#ffb380'), (1, '#ff6a00')], N=256)
norm = TwoSlopeNorm(vmin=0, vcenter=1, vmax=5)
grid_all = ok.pivot_table(index=['dataset', 'k', 'min_support'], columns='method', values='wall_s_median').reset_index()
grid_all['pf'] = grid_all[PF].min(axis=1)
grid_all['seeded'] = [swall.hyb_seeded.get((d, k, round(s, 7)), np.nan) for d, k, s in zip(grid_all.dataset, grid_all.k, grid_all.min_support)]
def lowest(col, k, base=None):
    base = base or ('fastapriori_fast' if k == 2 else 'seeded')
    q = grid_all[(grid_all.k == k)].dropna(subset=[base, col])
    q = q.loc[q.groupby('dataset').min_support.idxmin()].set_index('dataset')
    return q[col] / q[base]
fig, axes = plt.subplots(2, 3, figsize=(7.1, 4.65), layout='constrained')
fig.get_layout_engine().set(w_pad=.02, h_pad=.03, hspace=.02, wspace=.02)
regime = {}
stats_d = st.assign(density=st.d_avg / st.N)
bubble = lambda x: .3 * np.cbrt(x)
for r, (col, lab) in enumerate([('pf', 'PyFIM best'), ('fastapriori_classic', 'classic')]):
    for c, k in enumerate([2, 3, 4]):
        ax = axes[r, c]
        sp = lowest(col, k)
        z = stats_d.join(sp.rename('sp'), how='inner')
        ax.scatter(z.density, z.d_max, s=bubble(z.rows), c=z.sp, cmap=cmap, norm=norm, edgecolor='#000000', lw=.5, alpha=.92, zorder=3, clip_on=False)
        win = int((z.sp > 1).sum())
        regime[f'{col}_k{k}'] = dict(wins=win, n=len(z), speedup={name(d): round(float(v), 2) for d, v in z.sp.sort_values().items()})
        ax.set_title(f'vs {lab} ($k={k}$)', loc='center', fontsize=7, fontweight='bold', pad=3)
        ax.text(.97, .96, f'Faster: {win} of {len(z)}', transform=ax.transAxes, va='top', ha='right', fontsize=5.3, fontweight='bold', color=INK,
                bbox=dict(boxstyle='round,pad=.25', facecolor='#ffffff', edgecolor='#888888', lw=.4), zorder=5)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlim(5e-5, 5e-2)
        ax.set_ylim(10, 1e4)
        ax.xaxis.set_major_locator(FixedLocator([1e-4, 1e-3, 1e-2]))
        ax.yaxis.set_major_locator(FixedLocator([10, 1e2, 1e3, 1e4]))
        for a in (ax.xaxis, ax.yaxis):
            a.set_major_formatter(FuncFormatter(lambda v, _: sci(v)))
            a.set_minor_locator(NullLocator())
        square(ax)
        ax.tick_params(axis='both', labelsize=6, pad=2)
        ax.grid(True, which='major', color=GRID, lw=.5)
        ax.set_axisbelow(True)
        if r == 1:
            ax.set_xlabel('Density', fontsize=7)
        if c == 0:
            ax.set_ylabel(r'$d_{\max}$', fontsize=7)
N['regime'] = regime
cl_all = pd.concat([v for v in [lowest('fastapriori_classic', k) for k in [2, 3, 4]]])
N['regime_classic_over5'] = [int((cl_all > 5).sum()), len(cl_all), float(cl_all.max())]
kinds = sall[sall.status == 'oom'].stderr_tail
N['seeded_oom_kinds'] = dict(numpy=int(kinds.str.contains('ArrayMemoryError').sum()), rust=int(kinds.str.contains('memory allocation of').sum()), n=len(kinds))
z3 = stats_d.join(lowest('pf', 3).rename('sp'), how='inner')
N['regime_k3_density'] = {name(d): [round(float(r.density), 4), round(float(r.sp), 2)] for d, r in z3.sort_values('sp').iterrows()}
N['regime_anchor'] = {f'{col}_k{k}': dict(wins=int((lowest(col, k, 'fastapriori_fast') > 1).sum()), n=len(lowest(col, k, 'fastapriori_fast'))) for col in ['pf', 'fastapriori_classic'] for k in [3, 4]}
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cb = fig.colorbar(sm, ax=axes.ravel().tolist(), label='Speedup', fraction=.022, pad=.01, aspect=36)
cb.set_ticks([0, 1, 2, 3, 4, 5], labels=['0', '1', '2', '3', '4', r'$\geq$5'])
# one placement for all panels: density and d_max do not change with k or the comparator
OFF = {'Kosarak': (0, 5, 'center'), 'Online Retail': (0, 5, 'center'), 'BMS-WV1': (5, 0, 'left'), 'BMS-WV2': (4, 0, 'left'), 'Chainstore': (3, 9, 'left'),
       'Instacart': (7, 2, 'left'), 'Retail-BE': (1, 5, 'center'), 'T30.D2M': (5, 8, 'center'), 'T30.D50k': (0, -9, 'center'), 'T25.D5M': (7, -3, 'left'),
       'T20.D1M': (-6, -1, 'right'), 'T10.D100k': (5, -2, 'left'), 'T15.D50k.h': (8, -13, 'center'), 'T15.D50k.l': (2, 12, 'center'), 'Groceries': (2, 7, 'center'),
       'T12.D5k': (-11, -3, 'center'), 'T5.D25k': (-4, -1, 'right'), 'T3.D100k': (4, 0, 'left')}
LBL = {'BMS-WebView-1': 'BMS-WV1', 'BMS-WebView-2': 'BMS-WV2', 'Retail': 'Retail-BE', 'Retail (Belgian)': 'Retail-BE'}
for r, col in enumerate(['pf', 'fastapriori_classic']):
    for c, k in enumerate([2, 3, 4]):
        ax = axes[r, c]
        z = stats_d.join(lowest(col, k).rename('sp'), how='inner')
        for d, a in z.iterrows():
            lab = LBL.get(name(d), name(d))
            dx, dy, ha = OFF[lab]
            ax.annotate(lab, (a.density, a.d_max), xytext=(dx, dy), textcoords='offset points', fontsize=5.3, color=INK, ha=ha, va='center', zorder=4,
                        arrowprops=dict(arrowstyle='-', color=INK, lw=.35, shrinkA=0, shrinkB=0) if dy ** 2 + dx ** 2 > 140 else None)
axes[0, 0].legend(handles=[Line2D([], [], marker='o', color='none', mfc='#ffffff', mec='#000000', mew=.5, ms=np.sqrt(bubble(1e6)))], labels=['1M incidences'],
                  loc='upper left', fontsize=5.3, frameon=False, handletextpad=.1, borderpad=.25, borderaxespad=.1)
fig.savefig(FIG / 'regime_per_k.pdf')
plt.close(fig)


In [25]:
# ---- campaign table ------------------------------------------------------------------------------
def tally(base, files):
    tot = dict(att=0, ok=0, bad=0, skip=0, meas=0)
    for f in files:
        d = pd.read_csv(base / f'{f}.csv')
        s = d.groupby(['dataset', 'method', 'k', 'min_support']).status.agg(lambda x: 'ok' if (x == 'ok').all() else [v for v in x if v != 'ok'][0])
        tot['att'] += len(s); tot['ok'] += int((s == 'ok').sum()); tot['skip'] += int((s == 'skipped').sum())
        tot['bad'] += int(s.isin(['oom', 'timeout']).sum()); tot['meas'] += int(((d.status == 'ok') & ~d.warmup).sum())
    return tot
grid_t = dict(att=len(cov), ok=int((cov.probe_status == 'ok').sum()), bad=int((cov.probe_status != 'ok').sum()), skip=0, meas=int(ok.wall_s_n_ok.sum()))
CAMP = [('Main grid', '0.1.0.post1', '18 sparse', '5 supports, $k=2..9$', 'all but LCM; \\sys{} 1T, 8T', grid_t),
        ('Dense', '0.2.0', '5 dense', '6 supports, $k=2,3$', 'LCM, \\classic{}, Borgelt; 1T, 8T', tally(OCT, ['fimi_lite'])),
        ('Sweeps', '0.2.0', '16 generated', '$10^{-4}$, $k=2,3,4$', 'five externals, both paths', tally(OCT, ['scaling_D', 'scaling_T', 'scaling_dmax'])),
        ('Phase profile', '0.2.0 + timers', '6 of the grid', '$10^{-4}$, $k=3,4,5$', '\\anchorpath{}, Borgelt Eclat', tally(OCT, ['phaseprof'])),
        ('Standalone', '0.2.0', '6 of the grid', '$10^{-4}$, $k=2,3$', 'no pandas; Borgelt, LCM', tally(OCT, ['memory_boundary'])),
        ('Bound check', 'Borgelt only', '3 of the grid', '$10^{-4}$, $k=2,3$', 'bounded and unbounded', tally(OCT, ['a1_bounds'])),
        ('\\seeded{} grid', '0.2.0', '18 + 5 dense', '$10^{-4}$, $5\\cdot10^{-4}$, $k=3..9$; dense $k=3$', '\\seeded{} 1T, 8T', tally(NOV, ['seeded_grid', 'seeded_gaps_5e4', 'seeded_gaps_1e4', 'seeded_dense']))]
rows('tab_campaigns_rows.tex', [f"{a} & {b} & {c} & {d} & {e} & {integer(v['att'])} & {integer(v['ok'])} & {v['bad']} & {v['skip']} & {integer(v['meas'])} \\\\" for a, b, c, d, e, v in CAMP])
N['campaigns'] = {a: v for a, *_, v in CAMP}
N['campaigns_further'] = {k: sum(v[k] for a, *_, v in CAMP[1:]) for k in ['att', 'ok', 'bad', 'skip', 'meas']}
N['campaigns_further']['attempted'] = N['campaigns_further']['att'] - N['campaigns_further']['skip']

(GEN / 'numbers.json').write_text(json.dumps(N, indent=1, default=float), encoding='utf-8')


39349